In [25]:
customers_csv = """customer_id,name,city,age,signup_date
1,Amit,Hyderabad,28,2023-01-10
2,Priya,Bangalore,32,2023-02-12
3,Rahul,Mumbai,29,2023-03-14
4,Sneha,Delhi,35,2023-04-15
5,Arjun,Chennai,27,2023-05-11
6,Meera,Hyderabad,31,2023-06-10
7,Karan,Pune,33,2023-06-22
8,Neha,Delhi,30,2023-07-10
9,Divya,Bangalore,26,2023-07-15
10,Vikram,Mumbai,40,2023-08-01
11,Ritu,Hyderabad,34,2023-08-10
12,Sanjay,Delhi,38,2023-08-21
13,Naveen,Chennai,28,2023-09-01
14,Farhan,Mumbai,36,2023-09-10

15,Simran,Bangalore,25,2023-09-18
"""

In [26]:
products_csv = """product_id,product_name,category,price
101,Laptop,Electronics,75000
102,Headphones,Electronics,3000
103,Keyboard,Electronics,1500
104,Monitor,Electronics,12000
105,Office Chair,Furniture,7000
106,Desk,Furniture,15000
107,Smartphone,Electronics,40000
108,Notebook,Stationery,100
109,Pen,Stationery,20
110,Tablet,Electronics,30000
"""


In [27]:

orders_csv = """order_id,customer_id,order_date,status
1,1,2024-03-01,Delivered
2,2,2024-03-02,Delivered
3,3,2024-03-03,Cancelled
4,4,2024-03-04,Delivered
5,5,2024-03-05,Delivered
6,6,2024-03-06,Delivered
7,7,2024-03-07,Pending
8,8,2024-03-08,Delivered
9,9,2024-03-09,Delivered
10,10,2024-03-10,Delivered
"""


In [28]:

order_items_csv = """order_id,product_id,quantity
1,101,1
1,102,2
2,103,1
3,101,1
4,104,1
5,107,1
6,102,3
7,103,2
8,110,1
9,108,5
10,109,10
"""


In [29]:

payments_csv = """payment_id,order_id,payment_type,amount

1,1,Credit Card,81000
2,2,UPI,1500
3,3,Debit Card,75000
4,4,Credit Card,12000
5,5,UPI,40000
6,6,UPI,9000
7,7,Debit Card,3000
8,8,Credit Card,30000
9,9,UPI,500
10,10,UPI,200
"""



In [30]:
logs_txt = """login Amit
login Priya
view Rahul
purchase Amit
view Sneha
login Amit
purchase Priya
view Arjun
purchase Sneha
login Rahul
logout Amit
login Neha
purchase Neha
view Vikram
purchase Vikram
login Farhan
view Farhan
purchase Farhan
login Simran
purchase Simran
"""

In [31]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("ECommerceAnalytics").getOrCreate()
sc = spark.sparkContext

In [32]:
with open("customers.csv","w") as f:
  f.write(customers_csv)

In [33]:
with open("products.csv","w") as f:
  f.write(products_csv)

In [34]:
with open("orders.csv","w") as f:
  f.write(orders_csv)

In [35]:
with open("order_items.csv","w") as f:
  f.write(order_items_csv)

In [36]:
with open("payments.csv","w") as f:
  f.write(payments_csv)

In [37]:
with open("logs.txt","w") as f:
  f.write(logs_txt)

In [38]:
customers = spark.read.csv("customers.csv", header=True, inferSchema=True)

In [39]:
orders = spark.read.csv("orders.csv", header=True, inferSchema=True)

In [40]:
payments = spark.read.csv("payments.csv", header=True, inferSchema=True)

In [41]:
products = spark.read.csv("products.csv", header=True, inferSchema=True)

In [42]:
order_items = spark.read.csv("order_items.csv", header=True, inferSchema=True)

In [43]:
customers.show()

+-----------+------+---------+---+-----------+
|customer_id|  name|     city|age|signup_date|
+-----------+------+---------+---+-----------+
|          1|  Amit|Hyderabad| 28| 2023-01-10|
|          2| Priya|Bangalore| 32| 2023-02-12|
|          3| Rahul|   Mumbai| 29| 2023-03-14|
|          4| Sneha|    Delhi| 35| 2023-04-15|
|          5| Arjun|  Chennai| 27| 2023-05-11|
|          6| Meera|Hyderabad| 31| 2023-06-10|
|          7| Karan|     Pune| 33| 2023-06-22|
|          8|  Neha|    Delhi| 30| 2023-07-10|
|          9| Divya|Bangalore| 26| 2023-07-15|
|         10|Vikram|   Mumbai| 40| 2023-08-01|
|         11|  Ritu|Hyderabad| 34| 2023-08-10|
|         12|Sanjay|    Delhi| 38| 2023-08-21|
|         13|Naveen|  Chennai| 28| 2023-09-01|
|         14|Farhan|   Mumbai| 36| 2023-09-10|
|         15|Simran|Bangalore| 25| 2023-09-18|
+-----------+------+---------+---+-----------+



In [44]:
customers.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- signup_date: date (nullable = true)



In [45]:
print(f"Total Customers: {customers.count()}")

Total Customers: 15


In [46]:
customers.show(5)

+-----------+-----+---------+---+-----------+
|customer_id| name|     city|age|signup_date|
+-----------+-----+---------+---+-----------+
|          1| Amit|Hyderabad| 28| 2023-01-10|
|          2|Priya|Bangalore| 32| 2023-02-12|
|          3|Rahul|   Mumbai| 29| 2023-03-14|
|          4|Sneha|    Delhi| 35| 2023-04-15|
|          5|Arjun|  Chennai| 27| 2023-05-11|
+-----------+-----+---------+---+-----------+
only showing top 5 rows


In [47]:
customers.select("name","city").show()

+------+---------+
|  name|     city|
+------+---------+
|  Amit|Hyderabad|
| Priya|Bangalore|
| Rahul|   Mumbai|
| Sneha|    Delhi|
| Arjun|  Chennai|
| Meera|Hyderabad|
| Karan|     Pune|
|  Neha|    Delhi|
| Divya|Bangalore|
|Vikram|   Mumbai|
|  Ritu|Hyderabad|
|Sanjay|    Delhi|
|Naveen|  Chennai|
|Farhan|   Mumbai|
|Simran|Bangalore|
+------+---------+



In [48]:
print(f"Total Products: {products.count()}")

Total Products: 10


In [49]:
products.select("product_name","price").show()

+------------+-----+
|product_name|price|
+------------+-----+
|      Laptop|75000|
|  Headphones| 3000|
|    Keyboard| 1500|
|     Monitor|12000|
|Office Chair| 7000|
|        Desk|15000|
|  Smartphone|40000|
|    Notebook|  100|
|         Pen|   20|
|      Tablet|30000|
+------------+-----+



In [50]:
orders.show()

+--------+-----------+----------+---------+
|order_id|customer_id|order_date|   status|
+--------+-----------+----------+---------+
|       1|          1|2024-03-01|Delivered|
|       2|          2|2024-03-02|Delivered|
|       3|          3|2024-03-03|Cancelled|
|       4|          4|2024-03-04|Delivered|
|       5|          5|2024-03-05|Delivered|
|       6|          6|2024-03-06|Delivered|
|       7|          7|2024-03-07|  Pending|
|       8|          8|2024-03-08|Delivered|
|       9|          9|2024-03-09|Delivered|
|      10|         10|2024-03-10|Delivered|
+--------+-----------+----------+---------+



In [51]:
print(f"Total Orders: {orders.count()}")

Total Orders: 10


In [52]:
payments.show()

+----------+--------+------------+------+
|payment_id|order_id|payment_type|amount|
+----------+--------+------------+------+
|         1|       1| Credit Card| 81000|
|         2|       2|         UPI|  1500|
|         3|       3|  Debit Card| 75000|
|         4|       4| Credit Card| 12000|
|         5|       5|         UPI| 40000|
|         6|       6|         UPI|  9000|
|         7|       7|  Debit Card|  3000|
|         8|       8| Credit Card| 30000|
|         9|       9|         UPI|   500|
|        10|      10|         UPI|   200|
+----------+--------+------------+------+



In [53]:
from pyspark.sql import functions as F

In [54]:
customers.filter(F.col("city")=="Hyderabad").show()

+-----------+-----+---------+---+-----------+
|customer_id| name|     city|age|signup_date|
+-----------+-----+---------+---+-----------+
|          1| Amit|Hyderabad| 28| 2023-01-10|
|          6|Meera|Hyderabad| 31| 2023-06-10|
|         11| Ritu|Hyderabad| 34| 2023-08-10|
+-----------+-----+---------+---+-----------+



In [55]:
customers.filter(F.col("age")>30).show()

+-----------+------+---------+---+-----------+
|customer_id|  name|     city|age|signup_date|
+-----------+------+---------+---+-----------+
|          2| Priya|Bangalore| 32| 2023-02-12|
|          4| Sneha|    Delhi| 35| 2023-04-15|
|          6| Meera|Hyderabad| 31| 2023-06-10|
|          7| Karan|     Pune| 33| 2023-06-22|
|         10|Vikram|   Mumbai| 40| 2023-08-01|
|         11|  Ritu|Hyderabad| 34| 2023-08-10|
|         12|Sanjay|    Delhi| 38| 2023-08-21|
|         14|Farhan|   Mumbai| 36| 2023-09-10|
+-----------+------+---------+---+-----------+



In [56]:
products.filter(F.col("price")>10000).show()

+----------+------------+-----------+-----+
|product_id|product_name|   category|price|
+----------+------------+-----------+-----+
|       101|      Laptop|Electronics|75000|
|       104|     Monitor|Electronics|12000|
|       106|        Desk|  Furniture|15000|
|       107|  Smartphone|Electronics|40000|
|       110|      Tablet|Electronics|30000|
+----------+------------+-----------+-----+



In [57]:
products.filter(F.col("category")=="Electronics").show()

+----------+------------+-----------+-----+
|product_id|product_name|   category|price|
+----------+------------+-----------+-----+
|       101|      Laptop|Electronics|75000|
|       102|  Headphones|Electronics| 3000|
|       103|    Keyboard|Electronics| 1500|
|       104|     Monitor|Electronics|12000|
|       107|  Smartphone|Electronics|40000|
|       110|      Tablet|Electronics|30000|
+----------+------------+-----------+-----+



In [58]:
orders.filter(F.col("status")=="Delivered").show()

+--------+-----------+----------+---------+
|order_id|customer_id|order_date|   status|
+--------+-----------+----------+---------+
|       1|          1|2024-03-01|Delivered|
|       2|          2|2024-03-02|Delivered|
|       4|          4|2024-03-04|Delivered|
|       5|          5|2024-03-05|Delivered|
|       6|          6|2024-03-06|Delivered|
|       8|          8|2024-03-08|Delivered|
|       9|          9|2024-03-09|Delivered|
|      10|         10|2024-03-10|Delivered|
+--------+-----------+----------+---------+



In [59]:
orders.filter(F.col("status")=="Cancelled").show()

+--------+-----------+----------+---------+
|order_id|customer_id|order_date|   status|
+--------+-----------+----------+---------+
|       3|          3|2024-03-03|Cancelled|
+--------+-----------+----------+---------+



In [60]:
payments.filter(F.col("payment_type")=="UPI").show()

+----------+--------+------------+------+
|payment_id|order_id|payment_type|amount|
+----------+--------+------------+------+
|         2|       2|         UPI|  1500|
|         5|       5|         UPI| 40000|
|         6|       6|         UPI|  9000|
|         9|       9|         UPI|   500|
|        10|      10|         UPI|   200|
+----------+--------+------------+------+



In [61]:
customers.filter(F.col("city").isin("Bangalore","Delhi")).show()

+-----------+------+---------+---+-----------+
|customer_id|  name|     city|age|signup_date|
+-----------+------+---------+---+-----------+
|          2| Priya|Bangalore| 32| 2023-02-12|
|          4| Sneha|    Delhi| 35| 2023-04-15|
|          8|  Neha|    Delhi| 30| 2023-07-10|
|          9| Divya|Bangalore| 26| 2023-07-15|
|         12|Sanjay|    Delhi| 38| 2023-08-21|
|         15|Simran|Bangalore| 25| 2023-09-18|
+-----------+------+---------+---+-----------+



In [62]:
products.filter(F.col("price")<1000).show()

+----------+------------+----------+-----+
|product_id|product_name|  category|price|
+----------+------------+----------+-----+
|       108|    Notebook|Stationery|  100|
|       109|         Pen|Stationery|   20|
+----------+------------+----------+-----+



In [63]:
customers.filter(F.col("age").between(25,35)).show()

+-----------+------+---------+---+-----------+
|customer_id|  name|     city|age|signup_date|
+-----------+------+---------+---+-----------+
|          1|  Amit|Hyderabad| 28| 2023-01-10|
|          2| Priya|Bangalore| 32| 2023-02-12|
|          3| Rahul|   Mumbai| 29| 2023-03-14|
|          4| Sneha|    Delhi| 35| 2023-04-15|
|          5| Arjun|  Chennai| 27| 2023-05-11|
|          6| Meera|Hyderabad| 31| 2023-06-10|
|          7| Karan|     Pune| 33| 2023-06-22|
|          8|  Neha|    Delhi| 30| 2023-07-10|
|          9| Divya|Bangalore| 26| 2023-07-15|
|         11|  Ritu|Hyderabad| 34| 2023-08-10|
|         13|Naveen|  Chennai| 28| 2023-09-01|
|         15|Simran|Bangalore| 25| 2023-09-18|
+-----------+------+---------+---+-----------+



In [64]:
products_renamed = products.withColumnRenamed("price", "product_price")

In [65]:
customers_renamed = customers.withColumnRenamed("name", "customer_name")

In [66]:
customers.select("name", "age").show()

+------+---+
|  name|age|
+------+---+
|  Amit| 28|
| Priya| 32|
| Rahul| 29|
| Sneha| 35|
| Arjun| 27|
| Meera| 31|
| Karan| 33|
|  Neha| 30|
| Divya| 26|
|Vikram| 40|
|  Ritu| 34|
|Sanjay| 38|
|Naveen| 28|
|Farhan| 36|
|Simran| 25|
+------+---+



In [67]:
products.select("product_name", "category").show()

+------------+-----------+
|product_name|   category|
+------------+-----------+
|      Laptop|Electronics|
|  Headphones|Electronics|
|    Keyboard|Electronics|
|     Monitor|Electronics|
|Office Chair|  Furniture|
|        Desk|  Furniture|
|  Smartphone|Electronics|
|    Notebook| Stationery|
|         Pen| Stationery|
|      Tablet|Electronics|
+------------+-----------+



In [68]:
orders.select("order_id", "order_date").show()

+--------+----------+
|order_id|order_date|
+--------+----------+
|       1|2024-03-01|
|       2|2024-03-02|
|       3|2024-03-03|
|       4|2024-03-04|
|       5|2024-03-05|
|       6|2024-03-06|
|       7|2024-03-07|
|       8|2024-03-08|
|       9|2024-03-09|
|      10|2024-03-10|
+--------+----------+



In [69]:
payments.select("payment_type", "amount").show()

+------------+------+
|payment_type|amount|
+------------+------+
| Credit Card| 81000|
|         UPI|  1500|
|  Debit Card| 75000|
| Credit Card| 12000|
|         UPI| 40000|
|         UPI|  9000|
|  Debit Card|  3000|
| Credit Card| 30000|
|         UPI|   500|
|         UPI|   200|
+------------+------+



In [70]:
products.withColumn("price_in_lakhs", F.col("price") / 100000).show()

+----------+------------+-----------+-----+--------------+
|product_id|product_name|   category|price|price_in_lakhs|
+----------+------------+-----------+-----+--------------+
|       101|      Laptop|Electronics|75000|          0.75|
|       102|  Headphones|Electronics| 3000|          0.03|
|       103|    Keyboard|Electronics| 1500|         0.015|
|       104|     Monitor|Electronics|12000|          0.12|
|       105|Office Chair|  Furniture| 7000|          0.07|
|       106|        Desk|  Furniture|15000|          0.15|
|       107|  Smartphone|Electronics|40000|           0.4|
|       108|    Notebook| Stationery|  100|         0.001|
|       109|         Pen| Stationery|   20|        2.0E-4|
|       110|      Tablet|Electronics|30000|           0.3|
+----------+------------+-----------+-----+--------------+



In [71]:
customers.withColumn("age_group", F.when(F.col("age") < 30, "Young").otherwise("Adult")).show()

+-----------+------+---------+---+-----------+---------+
|customer_id|  name|     city|age|signup_date|age_group|
+-----------+------+---------+---+-----------+---------+
|          1|  Amit|Hyderabad| 28| 2023-01-10|    Young|
|          2| Priya|Bangalore| 32| 2023-02-12|    Adult|
|          3| Rahul|   Mumbai| 29| 2023-03-14|    Young|
|          4| Sneha|    Delhi| 35| 2023-04-15|    Adult|
|          5| Arjun|  Chennai| 27| 2023-05-11|    Young|
|          6| Meera|Hyderabad| 31| 2023-06-10|    Adult|
|          7| Karan|     Pune| 33| 2023-06-22|    Adult|
|          8|  Neha|    Delhi| 30| 2023-07-10|    Adult|
|          9| Divya|Bangalore| 26| 2023-07-15|    Young|
|         10|Vikram|   Mumbai| 40| 2023-08-01|    Adult|
|         11|  Ritu|Hyderabad| 34| 2023-08-10|    Adult|
|         12|Sanjay|    Delhi| 38| 2023-08-21|    Adult|
|         13|Naveen|  Chennai| 28| 2023-09-01|    Young|
|         14|Farhan|   Mumbai| 36| 2023-09-10|    Adult|
|         15|Simran|Bangalore| 

In [72]:
order_items.filter(F.col("quantity") > 2).show()

+--------+----------+--------+
|order_id|product_id|quantity|
+--------+----------+--------+
|       6|       102|       3|
|       9|       108|       5|
|      10|       109|      10|
+--------+----------+--------+



In [73]:
customers.orderBy("age").show()

+-----------+------+---------+---+-----------+
|customer_id|  name|     city|age|signup_date|
+-----------+------+---------+---+-----------+
|         15|Simran|Bangalore| 25| 2023-09-18|
|          9| Divya|Bangalore| 26| 2023-07-15|
|          5| Arjun|  Chennai| 27| 2023-05-11|
|          1|  Amit|Hyderabad| 28| 2023-01-10|
|         13|Naveen|  Chennai| 28| 2023-09-01|
|          3| Rahul|   Mumbai| 29| 2023-03-14|
|          8|  Neha|    Delhi| 30| 2023-07-10|
|          6| Meera|Hyderabad| 31| 2023-06-10|
|          2| Priya|Bangalore| 32| 2023-02-12|
|          7| Karan|     Pune| 33| 2023-06-22|
|         11|  Ritu|Hyderabad| 34| 2023-08-10|
|          4| Sneha|    Delhi| 35| 2023-04-15|
|         14|Farhan|   Mumbai| 36| 2023-09-10|
|         12|Sanjay|    Delhi| 38| 2023-08-21|
|         10|Vikram|   Mumbai| 40| 2023-08-01|
+-----------+------+---------+---+-----------+



In [74]:
customers.orderBy(F.col("age").asc()).show()

+-----------+------+---------+---+-----------+
|customer_id|  name|     city|age|signup_date|
+-----------+------+---------+---+-----------+
|         15|Simran|Bangalore| 25| 2023-09-18|
|          9| Divya|Bangalore| 26| 2023-07-15|
|          5| Arjun|  Chennai| 27| 2023-05-11|
|          1|  Amit|Hyderabad| 28| 2023-01-10|
|         13|Naveen|  Chennai| 28| 2023-09-01|
|          3| Rahul|   Mumbai| 29| 2023-03-14|
|          8|  Neha|    Delhi| 30| 2023-07-10|
|          6| Meera|Hyderabad| 31| 2023-06-10|
|          2| Priya|Bangalore| 32| 2023-02-12|
|          7| Karan|     Pune| 33| 2023-06-22|
|         11|  Ritu|Hyderabad| 34| 2023-08-10|
|          4| Sneha|    Delhi| 35| 2023-04-15|
|         14|Farhan|   Mumbai| 36| 2023-09-10|
|         12|Sanjay|    Delhi| 38| 2023-08-21|
|         10|Vikram|   Mumbai| 40| 2023-08-01|
+-----------+------+---------+---+-----------+



In [75]:
customers.orderBy(F.col("age").desc()).show()

+-----------+------+---------+---+-----------+
|customer_id|  name|     city|age|signup_date|
+-----------+------+---------+---+-----------+
|         10|Vikram|   Mumbai| 40| 2023-08-01|
|         12|Sanjay|    Delhi| 38| 2023-08-21|
|         14|Farhan|   Mumbai| 36| 2023-09-10|
|          4| Sneha|    Delhi| 35| 2023-04-15|
|         11|  Ritu|Hyderabad| 34| 2023-08-10|
|          7| Karan|     Pune| 33| 2023-06-22|
|          2| Priya|Bangalore| 32| 2023-02-12|
|          6| Meera|Hyderabad| 31| 2023-06-10|
|          8|  Neha|    Delhi| 30| 2023-07-10|
|          3| Rahul|   Mumbai| 29| 2023-03-14|
|          1|  Amit|Hyderabad| 28| 2023-01-10|
|         13|Naveen|  Chennai| 28| 2023-09-01|
|          5| Arjun|  Chennai| 27| 2023-05-11|
|          9| Divya|Bangalore| 26| 2023-07-15|
|         15|Simran|Bangalore| 25| 2023-09-18|
+-----------+------+---------+---+-----------+



In [76]:
products.orderBy(F.col("price").desc()).limit(5).show()

+----------+------------+-----------+-----+
|product_id|product_name|   category|price|
+----------+------------+-----------+-----+
|       101|      Laptop|Electronics|75000|
|       107|  Smartphone|Electronics|40000|
|       110|      Tablet|Electronics|30000|
|       106|        Desk|  Furniture|15000|
|       104|     Monitor|Electronics|12000|
+----------+------------+-----------+-----+



In [77]:
products.orderBy(F.col("price").asc()).limit(3).show()

+----------+------------+-----------+-----+
|product_id|product_name|   category|price|
+----------+------------+-----------+-----+
|       109|         Pen| Stationery|   20|
|       108|    Notebook| Stationery|  100|
|       103|    Keyboard|Electronics| 1500|
+----------+------------+-----------+-----+



In [78]:
orders.orderBy("order_date").show()

+--------+-----------+----------+---------+
|order_id|customer_id|order_date|   status|
+--------+-----------+----------+---------+
|       1|          1|2024-03-01|Delivered|
|       2|          2|2024-03-02|Delivered|
|       3|          3|2024-03-03|Cancelled|
|       4|          4|2024-03-04|Delivered|
|       5|          5|2024-03-05|Delivered|
|       6|          6|2024-03-06|Delivered|
|       7|          7|2024-03-07|  Pending|
|       8|          8|2024-03-08|Delivered|
|       9|          9|2024-03-09|Delivered|
|      10|         10|2024-03-10|Delivered|
+--------+-----------+----------+---------+



In [79]:
payments.orderBy("amount").show()

+----------+--------+------------+------+
|payment_id|order_id|payment_type|amount|
+----------+--------+------------+------+
|        10|      10|         UPI|   200|
|         9|       9|         UPI|   500|
|         2|       2|         UPI|  1500|
|         7|       7|  Debit Card|  3000|
|         6|       6|         UPI|  9000|
|         4|       4| Credit Card| 12000|
|         8|       8| Credit Card| 30000|
|         5|       5|         UPI| 40000|
|         3|       3|  Debit Card| 75000|
|         1|       1| Credit Card| 81000|
+----------+--------+------------+------+



In [80]:
payments.orderBy(F.col("amount").desc()).limit(3).show()

+----------+--------+------------+------+
|payment_id|order_id|payment_type|amount|
+----------+--------+------------+------+
|         1|       1| Credit Card| 81000|
|         3|       3|  Debit Card| 75000|
|         5|       5|         UPI| 40000|
+----------+--------+------------+------+



In [81]:
customers.orderBy("city").show()

+-----------+------+---------+---+-----------+
|customer_id|  name|     city|age|signup_date|
+-----------+------+---------+---+-----------+
|          2| Priya|Bangalore| 32| 2023-02-12|
|          9| Divya|Bangalore| 26| 2023-07-15|
|         15|Simran|Bangalore| 25| 2023-09-18|
|          5| Arjun|  Chennai| 27| 2023-05-11|
|         13|Naveen|  Chennai| 28| 2023-09-01|
|          4| Sneha|    Delhi| 35| 2023-04-15|
|          8|  Neha|    Delhi| 30| 2023-07-10|
|         12|Sanjay|    Delhi| 38| 2023-08-21|
|          1|  Amit|Hyderabad| 28| 2023-01-10|
|          6| Meera|Hyderabad| 31| 2023-06-10|
|         11|  Ritu|Hyderabad| 34| 2023-08-10|
|          3| Rahul|   Mumbai| 29| 2023-03-14|
|         10|Vikram|   Mumbai| 40| 2023-08-01|
|         14|Farhan|   Mumbai| 36| 2023-09-10|
|          7| Karan|     Pune| 33| 2023-06-22|
+-----------+------+---------+---+-----------+



In [82]:
products.orderBy("category").show()

+----------+------------+-----------+-----+
|product_id|product_name|   category|price|
+----------+------------+-----------+-----+
|       101|      Laptop|Electronics|75000|
|       102|  Headphones|Electronics| 3000|
|       103|    Keyboard|Electronics| 1500|
|       104|     Monitor|Electronics|12000|
|       107|  Smartphone|Electronics|40000|
|       110|      Tablet|Electronics|30000|
|       105|Office Chair|  Furniture| 7000|
|       106|        Desk|  Furniture|15000|
|       108|    Notebook| Stationery|  100|
|       109|         Pen| Stationery|   20|
+----------+------------+-----------+-----+



In [83]:
payments = spark.read.csv("payments.csv", header=True, inferSchema=True)

In [84]:
payments.select(F.sum("amount")).show()
payments.select(F.avg("amount")).show()
payments.select(F.max("amount")).show()
payments.select(F.min("amount")).show()

+-----------+
|sum(amount)|
+-----------+
|     252200|
+-----------+

+-----------+
|avg(amount)|
+-----------+
|    25220.0|
+-----------+

+-----------+
|max(amount)|
+-----------+
|      81000|
+-----------+

+-----------+
|min(amount)|
+-----------+
|        200|
+-----------+



In [85]:
customers.groupBy("city").count().show()
products.groupBy("category").count().show()
products.groupBy("category").avg("price").show()
order_items.groupBy("product_id").sum("quantity").show()
customers.select(F.avg("age")).show()
print(f"Total Orders: {orders.count()}")

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    3|
|  Chennai|    2|
|   Mumbai|    3|
|     Pune|    1|
|    Delhi|    3|
|Hyderabad|    3|
+---------+-----+

+-----------+-----+
|   category|count|
+-----------+-----+
| Stationery|    2|
|Electronics|    6|
|  Furniture|    2|
+-----------+-----+

+-----------+------------------+
|   category|        avg(price)|
+-----------+------------------+
| Stationery|              60.0|
|Electronics|26916.666666666668|
|  Furniture|           11000.0|
+-----------+------------------+

+----------+-------------+
|product_id|sum(quantity)|
+----------+-------------+
|       108|            5|
|       101|            2|
|       103|            3|
|       107|            1|
|       102|            5|
|       109|           10|
|       110|            1|
|       104|            1|
+----------+-------------+

+------------------+
|          avg(age)|
+------------------+
|31.466666666666665|
+------------------+

Total Orders: 1

In [86]:
orders_cust = orders.join(customers, "customer_id")

In [87]:
orders_cust.select("name", "status").show()

+------+---------+
|  name|   status|
+------+---------+
|  Amit|Delivered|
| Priya|Delivered|
| Rahul|Cancelled|
| Sneha|Delivered|
| Arjun|Delivered|
| Meera|Delivered|
| Karan|  Pending|
|  Neha|Delivered|
| Divya|Delivered|
|Vikram|Delivered|
+------+---------+



In [88]:
orders_items = orders.join(order_items, "order_id")

In [89]:
items_products = order_items.join(products, "product_id")

In [90]:
items_products.withColumn("revenue", F.col("quantity") * F.col("price")).show()

+----------+--------+--------+------------+-----------+-----+-------+
|product_id|order_id|quantity|product_name|   category|price|revenue|
+----------+--------+--------+------------+-----------+-----+-------+
|       101|       3|       1|      Laptop|Electronics|75000|  75000|
|       101|       1|       1|      Laptop|Electronics|75000|  75000|
|       102|       6|       3|  Headphones|Electronics| 3000|   9000|
|       102|       1|       2|  Headphones|Electronics| 3000|   6000|
|       103|       7|       2|    Keyboard|Electronics| 1500|   3000|
|       103|       2|       1|    Keyboard|Electronics| 1500|   1500|
|       104|       4|       1|     Monitor|Electronics|12000|  12000|
|       107|       5|       1|  Smartphone|Electronics|40000|  40000|
|       108|       9|       5|    Notebook| Stationery|  100|    500|
|       109|      10|      10|         Pen| Stationery|   20|    200|
|       110|       8|       1|      Tablet|Electronics|30000|  30000|
+----------+--------

In [91]:
full_join = orders.join(customers, "customer_id").join(order_items, "order_id").join(products, "product_id")

In [92]:
full_join.select("name", "product_name", "quantity").show()

+------+------------+--------+
|  name|product_name|quantity|
+------+------------+--------+
|  Amit|  Headphones|       2|
|  Amit|      Laptop|       1|
| Priya|    Keyboard|       1|
| Rahul|      Laptop|       1|
| Sneha|     Monitor|       1|
| Arjun|  Smartphone|       1|
| Meera|  Headphones|       3|
| Karan|    Keyboard|       2|
|  Neha|      Tablet|       1|
| Divya|    Notebook|       5|
|Vikram|         Pen|      10|
+------+------------+--------+



In [93]:
full_join.groupBy("order_id").agg(F.sum(F.col("quantity") * F.col("price")).alias("order_revenue")).show()
full_join.groupBy("product_name").agg(F.sum(F.col("quantity") * F.col("price")).alias("product_revenue")).show()
full_join.groupBy("name").agg(F.sum(F.col("quantity") * F.col("price")).alias("customer_revenue")).show()

+--------+-------------+
|order_id|order_revenue|
+--------+-------------+
|       1|        81000|
|       6|         9000|
|       3|        75000|
|       5|        40000|
|       9|          500|
|       4|        12000|
|       8|        30000|
|       7|         3000|
|      10|          200|
|       2|         1500|
+--------+-------------+

+------------+---------------+
|product_name|product_revenue|
+------------+---------------+
|         Pen|            200|
|      Laptop|         150000|
|    Notebook|            500|
|      Tablet|          30000|
|    Keyboard|           4500|
|  Smartphone|          40000|
|     Monitor|          12000|
|  Headphones|          15000|
+------------+---------------+

+------+----------------+
|  name|customer_revenue|
+------+----------------+
| Divya|             500|
| Meera|            9000|
| Sneha|           12000|
| Priya|            1500|
|Vikram|             200|
| Rahul|           75000|
| Arjun|           40000|
|  Amit|        

In [94]:
orders.groupBy("customer_id").count().show()

+-----------+-----+
|customer_id|count|
+-----------+-----+
|          1|    1|
|          6|    1|
|          3|    1|
|          5|    1|
|          9|    1|
|          4|    1|
|          8|    1|
|          7|    1|
|         10|    1|
|          2|    1|
+-----------+-----+



In [95]:
orders_cust.groupBy("city").count().show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    2|
|  Chennai|    1|
|   Mumbai|    2|
|     Pune|    1|
|    Delhi|    2|
|Hyderabad|    2|
+---------+-----+



In [96]:
full_join.groupBy("product_id").agg(F.sum(F.col("quantity") * F.col("price")).alias("rev")).show()

+----------+------+
|product_id|   rev|
+----------+------+
|       108|   500|
|       101|150000|
|       103|  4500|
|       107| 40000|
|       102| 15000|
|       109|   200|
|       110| 30000|
|       104| 12000|
+----------+------+



In [97]:
full_join.groupBy("category").agg(F.sum(F.col("quantity") * F.col("price")).alias("rev")).show()

+-----------+------+
|   category|   rev|
+-----------+------+
| Stationery|   700|
|Electronics|251500|
+-----------+------+



In [98]:
full_join.groupBy("city").agg(F.sum(F.col("quantity") * F.col("price")).alias("rev")).show()

+---------+-----+
|     city|  rev|
+---------+-----+
|Bangalore| 2000|
|  Chennai|40000|
|   Mumbai|75200|
|     Pune| 3000|
|    Delhi|42000|
|Hyderabad|90000|
+---------+-----+



In [99]:
full_join.groupBy("customer_id").agg(F.sum(F.col("quantity") * F.col("price")).alias("rev")).show()
full_join.groupBy("name").agg(F.sum(F.col("quantity") * F.col("price")).alias("rev")).orderBy(F.desc("rev")).limit(5).show()
full_join.groupBy("product_name").agg(F.sum(F.col("quantity") * F.col("price")).alias("rev")).orderBy(F.desc("rev")).limit(3).show()

+-----------+-----+
|customer_id|  rev|
+-----------+-----+
|          1|81000|
|          6| 9000|
|          3|75000|
|          5|40000|
|          9|  500|
|          4|12000|
|          8|30000|
|          7| 3000|
|         10|  200|
|          2| 1500|
+-----------+-----+

+-----+-----+
| name|  rev|
+-----+-----+
| Amit|81000|
|Rahul|75000|
|Arjun|40000|
| Neha|30000|
|Sneha|12000|
+-----+-----+

+------------+------+
|product_name|   rev|
+------------+------+
|      Laptop|150000|
|  Smartphone| 40000|
|      Tablet| 30000|
+------------+------+



In [100]:
win_price = Window.orderBy(F.desc("price"))
products.withColumn("rank", F.rank().over(win_price)).show()
win_cat = Window.partitionBy("category").orderBy(F.desc("price"))
products.withColumn("cat_rank", F.rank().over(win_cat)).show()
win_signup = Window.orderBy("signup_date")
customers.withColumn("row_num", F.row_number().over(win_signup)).show()

+----------+------------+-----------+-----+----+
|product_id|product_name|   category|price|rank|
+----------+------------+-----------+-----+----+
|       101|      Laptop|Electronics|75000|   1|
|       107|  Smartphone|Electronics|40000|   2|
|       110|      Tablet|Electronics|30000|   3|
|       106|        Desk|  Furniture|15000|   4|
|       104|     Monitor|Electronics|12000|   5|
|       105|Office Chair|  Furniture| 7000|   6|
|       102|  Headphones|Electronics| 3000|   7|
|       103|    Keyboard|Electronics| 1500|   8|
|       108|    Notebook| Stationery|  100|   9|
|       109|         Pen| Stationery|   20|  10|
+----------+------------+-----------+-----+----+

+----------+------------+-----------+-----+--------+
|product_id|product_name|   category|price|cat_rank|
+----------+------------+-----------+-----+--------+
|       101|      Laptop|Electronics|75000|       1|
|       107|  Smartphone|Electronics|40000|       2|
|       110|      Tablet|Electronics|30000|     

In [101]:
cust_rev = full_join.groupBy("customer_id").agg(F.sum(F.col("quantity") * F.col("price")).alias("total_rev"))
cust_rev.withColumn("rank", F.rank().over(Window.orderBy(F.desc("total_rev")))).show()
cust_rev.withColumn("dense_rank", F.dense_rank().over(Window.orderBy(F.desc("total_rev")))).show()

+-----------+---------+----+
|customer_id|total_rev|rank|
+-----------+---------+----+
|          1|    81000|   1|
|          3|    75000|   2|
|          5|    40000|   3|
|          8|    30000|   4|
|          4|    12000|   5|
|          6|     9000|   6|
|          7|     3000|   7|
|          2|     1500|   8|
|          9|      500|   9|
|         10|      200|  10|
+-----------+---------+----+

+-----------+---------+----------+
|customer_id|total_rev|dense_rank|
+-----------+---------+----------+
|          1|    81000|         1|
|          3|    75000|         2|
|          5|    40000|         3|
|          8|    30000|         4|
|          4|    12000|         5|
|          6|     9000|         6|
|          7|     3000|         7|
|          2|     1500|         8|
|          9|      500|         9|
|         10|      200|        10|
+-----------+---------+----------+



In [102]:
full_join.withColumn("rn", F.row_number().over(Window.partitionBy("city").orderBy(F.desc(F.col("quantity") * F.col("price"))))).filter("rn=1").show()
full_join.withColumn("rn", F.row_number().over(Window.partitionBy("category").orderBy(F.desc(F.col("quantity") * F.col("price"))))).filter("rn=1").show()

+----------+--------+-----------+----------+---------+-----+---------+---+-----------+--------+------------+-----------+-----+---+
|product_id|order_id|customer_id|order_date|   status| name|     city|age|signup_date|quantity|product_name|   category|price| rn|
+----------+--------+-----------+----------+---------+-----+---------+---+-----------+--------+------------+-----------+-----+---+
|       103|       2|          2|2024-03-02|Delivered|Priya|Bangalore| 32| 2023-02-12|       1|    Keyboard|Electronics| 1500|  1|
|       107|       5|          5|2024-03-05|Delivered|Arjun|  Chennai| 27| 2023-05-11|       1|  Smartphone|Electronics|40000|  1|
|       110|       8|          8|2024-03-08|Delivered| Neha|    Delhi| 30| 2023-07-10|       1|      Tablet|Electronics|30000|  1|
|       101|       1|          1|2024-03-01|Delivered| Amit|Hyderabad| 28| 2023-01-10|       1|      Laptop|Electronics|75000|  1|
|       101|       3|          3|2024-03-03|Cancelled|Rahul|   Mumbai| 29| 2023-03-

In [103]:
logs = sc.textFile("logs.txt")
print(f"Log Count: {logs.count()}")
usernames = logs.map(lambda x: x.split(" ")[1])
actions = logs.map(lambda x: x.split(" ")[0])
print(f"Unique Users: {usernames.distinct().collect()}")
print(f"Logins: {logs.filter(lambda x: 'login' in x).count()}")
print(f"Purchases: {logs.filter(lambda x: 'purchase' in x).count()}")
print(f"Views: {logs.filter(lambda x: 'view' in x).count()}")
user_pairs = usernames.map(lambda x: (x, 1))
activity_count = user_pairs.reduceByKey(lambda a, b: a + b)
print(f"Most Active: {activity_count.max(lambda x: x[1])}")

Log Count: 20
Unique Users: ['Amit', 'Vikram', 'Simran', 'Priya', 'Rahul', 'Sneha', 'Arjun', 'Neha', 'Farhan']
Logins: 7
Purchases: 7
Views: 5
Most Active: ('Amit', 4)


In [104]:
customers.createOrReplaceTempView("customers")
orders.createOrReplaceTempView("orders")
products.createOrReplaceTempView("products")
payments.createOrReplaceTempView("payments")
order_items.createOrReplaceTempView("order_items")

In [106]:
spark.sql("SELECT * FROM customers").show()
spark.sql("SELECT * FROM customers WHERE age > 30").show()
spark.sql("SELECT city, count(*) FROM customers GROUP BY city").show()
spark.sql("SELECT city, avg(age) FROM customers GROUP BY city").show()
spark.sql("SELECT * FROM customers c JOIN orders o ON c.customer_id = o.customer_id").show()
spark.sql("SELECT customer_id, count(*) FROM orders GROUP BY customer_id").show()
spark.sql("SELECT * FROM order_items oi JOIN products p ON oi.product_id = p.product_id").show()
spark.sql("SELECT sum(amount) FROM payments").show()
spark.sql("""SELECT p.product_name, sum(oi.quantity * p.price) as rev
             FROM order_items oi JOIN products p ON oi.product_id = p.product_id
             GROUP BY p.product_name ORDER BY rev DESC LIMIT 1""").show()

+-----------+------+---------+---+-----------+
|customer_id|  name|     city|age|signup_date|
+-----------+------+---------+---+-----------+
|          1|  Amit|Hyderabad| 28| 2023-01-10|
|          2| Priya|Bangalore| 32| 2023-02-12|
|          3| Rahul|   Mumbai| 29| 2023-03-14|
|          4| Sneha|    Delhi| 35| 2023-04-15|
|          5| Arjun|  Chennai| 27| 2023-05-11|
|          6| Meera|Hyderabad| 31| 2023-06-10|
|          7| Karan|     Pune| 33| 2023-06-22|
|          8|  Neha|    Delhi| 30| 2023-07-10|
|          9| Divya|Bangalore| 26| 2023-07-15|
|         10|Vikram|   Mumbai| 40| 2023-08-01|
|         11|  Ritu|Hyderabad| 34| 2023-08-10|
|         12|Sanjay|    Delhi| 38| 2023-08-21|
|         13|Naveen|  Chennai| 28| 2023-09-01|
|         14|Farhan|   Mumbai| 36| 2023-09-10|
|         15|Simran|Bangalore| 25| 2023-09-18|
+-----------+------+---------+---+-----------+

+-----------+------+---------+---+-----------+
|customer_id|  name|     city|age|signup_date|
+-----------

Final Capstone Mini Project



In [107]:
final_df = orders.join(customers, "customer_id") \
                 .join(order_items, "order_id") \
                 .join(products, "product_id") \
                 .join(payments, "order_id")

In [108]:
final_df = final_df.withColumn("item_revenue", F.col("quantity") * F.col("price"))

In [109]:
final_df.groupBy("name").sum("item_revenue").orderBy(F.desc("sum(item_revenue)")).show(5)
final_df.groupBy("product_name").sum("item_revenue").orderBy(F.desc("sum(item_revenue)")).show(5)

+-----+-----------------+
| name|sum(item_revenue)|
+-----+-----------------+
| Amit|            81000|
|Rahul|            75000|
|Arjun|            40000|
| Neha|            30000|
|Sneha|            12000|
+-----+-----------------+
only showing top 5 rows
+------------+-----------------+
|product_name|sum(item_revenue)|
+------------+-----------------+
|      Laptop|           150000|
|  Smartphone|            40000|
|      Tablet|            30000|
|  Headphones|            15000|
|     Monitor|            12000|
+------------+-----------------+
only showing top 5 rows


In [110]:
final_df.groupBy("city").sum("item_revenue").show()

+---------+-----------------+
|     city|sum(item_revenue)|
+---------+-----------------+
|Bangalore|             2000|
|  Chennai|            40000|
|   Mumbai|            75200|
|     Pune|             3000|
|    Delhi|            42000|
|Hyderabad|            90000|
+---------+-----------------+



In [111]:
final_df.groupBy("name").agg(F.sum("item_revenue").alias("total")) \
        .withColumn("rank", F.rank().over(Window.orderBy(F.desc("total")))).show()

+------+-----+----+
|  name|total|rank|
+------+-----+----+
|  Amit|81000|   1|
| Rahul|75000|   2|
| Arjun|40000|   3|
|  Neha|30000|   4|
| Sneha|12000|   5|
| Meera| 9000|   6|
| Karan| 3000|   7|
| Priya| 1500|   8|
| Divya|  500|   9|
|Vikram|  200|  10|
+------+-----+----+



In [112]:
final_df.write.mode("overwrite").csv("ecommerce_report", header=True)